# What the 0.94650 actually is — six measurements

The top of the Code tab reads 0.94649–0.94650, four ten-thousandths above where most of us
stalled. I pulled every notebook at that score and measured what each ingredient is worth.
The short answer is less exciting than the titles, and more useful.

| # | question | measured answer |
|---|---|---|
| 1 | Are the 0.94650 notebooks different blends? | No. The ones I checked are the same file: Spearman **1.000000** against [Taeyang's lexsort](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master). |
| 2 | Where did the jump from 0.94645 come from? | One file: [jazivxt's](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) `submission_latest_best.csv`, **0.94649 on its own**. |
| 3 | Is tie-breaking the trick? | Barely. The anchor has 16,846 ties; its author reports 0.94649 with and without the lexsort. |
| 4 | Are the four "boundary" rules real? | Yes on this competition's train (393/393, 0/1,257, 0/186, 0/7,157) and **not** on the original data. |
| 5 | What are they worth? | **+0.57e-5** AUC on honest out-of-fold predictions, positive in **10 / 10** folds. Real, and half a digit. |
| 6 | Does an independent model push it higher? | Not this one. +20% → 0.94649, +30% → 0.94648, −5% and −10% → 0.94650. |

Row 6 is the one I paid for in submissions. My six-view ensemble lifted the old 0.94644 blend to
0.94645; the same ensemble on the 0.94650 base *hurts*, monotonically. Against a 0.94644 base it was
0.00005 weaker than the base; against 0.94650 it is 0.00011 weaker, and diversity only pays near parity.

The submission here is the 0.94650 base with 5% of that ensemble subtracted, the four rules and the
lexsort re-applied — scored **0.94650**, identical ordering to the file I submitted.

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

ROOT = "/kaggle/input"
ID, TARGET = "id", "Will_Buy_EV"

def find_one(*pats):
    for p in pats:
        h = sorted(glob.glob(f"{ROOT}/**/{p}", recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(" | ".join(pats))

def rk(v):
    return rankdata(v, method="average") / len(v)

train = pd.read_csv(find_one("playground-series-s6e9/train.csv", "train.csv"))
test = pd.read_csv(find_one("playground-series-s6e9/test.csv", "test.csv")).sort_values(ID).reset_index(drop=True)
orig = pd.read_csv(find_one("EV_Adoption_and_Range_Anxiety_Dataset.csv"))
ids, N = test[ID].to_numpy(), len(test)

def load_pred(*pats, col=None):
    p = find_one(*pats)
    d = pd.read_csv(p).sort_values(ID)
    assert np.array_equal(d[ID].to_numpy(), ids), p
    print("  ", p)
    return d[col or [c for c in d.columns if c != ID][0]].to_numpy(dtype=float)

print("sources:")
lex = load_pred("s6e9-094649-multi-paradigm-lexsort-master/submission.csv")          # 0.94650
anchor = load_pred("submission_latest_best.csv")                                       # 0.94649
realmlp = load_pred("ps-s6-e9-realmlp-pytorch/submission.csv")
six_test = pd.read_csv(find_one("test_six_views.csv")).sort_values(ID).reset_index(drop=True)
six_oof = pd.read_csv(find_one("oof_six_views.csv"))

## 1–3. One file, one anchor, sixteen thousand ties

A score that ties at the top across many notebooks is usually one set of predictions. Spearman
between two submissions is the right check: AUC only sees the ordering.

In [ ]:
ties = lambda v: len(v) - len(np.unique(v))
print(f"ties in jazivxt anchor (0.94649) : {ties(anchor):,}")
print(f"ties in lexsort output (0.94650) : {ties(lex):,}")
print(f"spearman(lexsort, anchor)        : {spearmanr(lex, anchor).statistic:.6f}")
print(f"spearman(lexsort, RealMLP)       : {spearmanr(lex, realmlp).statistic:.6f}")

The anchor is 0.94649 by itself; the lexsort author's own log (linked above) shows the anchor with
RealMLP tie-breaking also at 0.94649. The Spearman between lexsort and its anchor is lower than a
90% blend would suggest because the boundary rules below throw 3,733 test rows to the extremes.

For the repackages I compared the downloaded outputs directly (not re-run here):
chinzorigtganbat, rohitt94 and nathaliach against lexsort all read Spearman 1.000000; nina2025's
h-blend (5) reads 0.998653 and was LB-checked by talhatursun at 70/30 with lexsort: still 0.94650.

## 4. The four rules, checked on the data they claim

In [ ]:
def rule_masks(d):
    inc = pd.to_numeric(d.Annual_Income_USD, errors="coerce").to_numpy()
    km = pd.to_numeric(d.Daily_Commute_km, errors="coerce").to_numpy()
    env1 = pd.to_numeric(d.Environmental_Concern_Level, errors="coerce").to_numpy() == 1
    no_sub = d.Subsidy_Available.astype(str).to_numpy() == "No"
    anx_mh = d.Range_Anxiety_Level.isin(["Medium", "High"]).to_numpy()
    return {
        "income >= 170,537          (+10)": (inc >= 170537, +10.0),
        "31,004 <= income <= 41,970 (-10)": ((inc >= 31004) & (inc <= 41970), -10.0),
        "commute >= 83 km           (-5)":  (km >= 83, -5.0),
        "30k & no subsidy & (env1 | anxiety M/H) (-5)": ((inc == 30000) & no_sub & (env1 | anx_mh), -5.0),
    }

y_tr = (train[TARGET] == "Yes").astype(int).to_numpy()
y_or = (orig[TARGET].astype(str) == "Yes").astype(int).to_numpy()
m_tr, m_or, m_te = rule_masks(train), rule_masks(orig), rule_masks(test)
rows = []
for k in m_tr:
    a, b, c = m_tr[k][0], m_or[k][0], m_te[k][0]
    rows.append([k, a.sum(), y_tr[a].mean(), b.sum(), y_or[b].mean() if b.sum() else np.nan, c.sum()])
pd.DataFrame(rows, columns=["rule", "comp train rows", "comp buy rate", "orig rows", "orig buy rate", "test rows"]).round(4)

Pure on 668,665 competition rows, clearly not pure on the 10,000-row original. These are cells the
generator made deterministic, which is exactly the kind of artefact that holds on the test set.

## 5. What forcing them is worth, on honest out-of-fold predictions

The rules are fixed, so no mining happens inside the folds; the question is only how much a
reasonable model already gets right in these cells. I use the `ensemble` column of my
[six-view OOF library](https://www.kaggle.com/datasets/megayak/s6e9-six-feature-views-oof-library)
(OOF AUC 0.946345) and its stored fold ids.

In [ ]:
oof = six_oof.sort_values(ID)
train_s = train.sort_values(ID).reset_index(drop=True)
assert np.array_equal(oof[ID].to_numpy(), train_s[ID].to_numpy())
y = oof[TARGET].to_numpy(); p = oof["ensemble"].to_numpy(); fold = oof["fold"].to_numpy()
masks = rule_masks(train_s)

def fold_gain(shift):
    return np.array([roc_auc_score(y[fold == f], p[fold == f] + shift[fold == f])
                     - roc_auc_score(y[fold == f], p[fold == f]) for f in range(10)])

order = np.argsort(np.argsort(p)) / len(p)
out = []
for k, (m, s) in masks.items():
    g = fold_gain(np.where(m, s, 0.0))
    out.append([k, m.sum(), np.median(order[m]), g.mean() * 1e5, (g > 0).sum()])
g = fold_gain(sum(np.where(m, s, 0.0) for m, s in masks.values()))
out.append(["all four", sum(m.sum() for m, _ in masks.values()), np.nan, g.mean() * 1e5, (g > 0).sum()])
pd.DataFrame(out, columns=["rule", "train rows", "model rank pct (median)", "gain x1e-5", "folds up / 10"]).round(3)

The model already puts the 30k cell in the bottom 2% and the upper cliff at the top 0.1%, so forcing
them moves little. The dead-income zone is the one with room (some rows sit as high as the 70th
percentile) and it carries most of the gain. All four together: about half a unit in the fifth
decimal, consistent in every fold.

## 6. An independent ensemble on the 0.94650 base

These are leaderboard numbers, one submission each, same base, same rules and lexsort re-applied.

| construction | public LB |
|---|---|
| lexsort alone | 0.94650 |
| lexsort + 20% six-view ensemble | 0.94649 |
| lexsort + 30% six-view ensemble | 0.94648 |
| jazivxt anchor (lexsorted) + 20% ensemble | 0.94649 |
| **lexsort − 5% ensemble** | **0.94650** |
| lexsort − 10% ensemble | 0.94650 |

Monotone on the positive side, flat on the negative side: the base sits at the top of a concave curve
with respect to this ensemble. The same ensemble *added* +0.00001 to the older 0.94644 base. The
difference is parity — 0.00005 behind that base, 0.00011 behind this one.

## Submission

In [ ]:
W = {"A_lgbm_triple_te_digits_3seed": .2, "B_xgb_on_A_features": .2, "C_no_digits_windows_lift_sm2_30_300": .1,
     "D_no_exact_key_ladder_windows": .2, "E_ladder25_250_2500_lift_sm5_50_500": .1, "F_exact_rate_as_init_score": .2}
assert np.array_equal(six_test[ID].to_numpy(), ids)
ensemble = rk(sum(w * rk(six_test[c].to_numpy(dtype=float)) for c, w in W.items()))

shift = sum(np.where(m, s, 0.0) for m, s in m_te.values())

def lexrank(primary, secondary):
    o = np.lexsort((secondary, primary))
    r = np.empty(N); r[o] = np.arange(1, N + 1)
    return (r - 0.5) / N

final = lexrank(rk(lex) - 0.05 * ensemble + shift, realmlp)
assert len(np.unique(final)) == N and np.isfinite(final).all()
print(f"spearman(final, lexsort 0.94650) = {spearmanr(final, lex).statistic:.6f}")
pd.DataFrame({ID: ids, TARGET: final}).to_csv("submission.csv", index=False)
print("wrote submission.csv")

## What I take from this

The leaderboard above 0.94645 was not moved by a modelling idea. It was moved by one strong file and
is held there by a few thousand rows the generator made certain. The honest-OOF value of those rows is
half a digit, and the independent ensemble that worked one step down does not work here.

@tilii7 put the general point plainly in the discussion: re-blending public outputs fits the public
board, and only ensembling from OOF is reliable. The table in section 6 is my own evidence for that —
it is why this notebook measures instead of stacking another public file, and why the six-view
library it uses ships its OOF and fold ids.

Where I would look next, in order of how much I believe it:
1. A model that is **at parity with 0.94650 on its own** and built differently. Nothing public is.
2. Pure cells beyond these four — but see my follow-up: naive mining finds twelve thousand "pure" cells
   and forcing them costs −0.0024. The filter has to be significance, not purity.

---
Credit: [@taeyangg4](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master) for the lexsort
blend and the four rules, [@jazivxt](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) for the anchor,
[@yekenot](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch) for RealMLP,
[@talhatursun](https://www.kaggle.com/code/talhatursun/s6e9-daily-rank-average-ensemble) for the lexsort / h-blend (5) check,
[@itzzomkar](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety) for the original data.
The measurements and the six-view ensemble are mine. If this saved you submissions, an upvote helps.